### Mount Google Drive
To access the files in your shared folder, we first need to mount Google Drive to this notebook.

In [1]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive')

Mounted at /content/drive


### Load and Combine the 4 Excel Files
Replace the folder path below with the path to the folder containing your 4 Excel files (e.g., if you added a shortcut to your own Drive).

In [2]:
import pandas as pd
import os
import glob

# 1. Add the shared folder as a shortcut in your Google Drive (e.g., named 'LendingRiskFolder')
# 2. Update the folder name below to match your shortcut's name exactly
folder_name = 'LendingRiskFolder'

folder_path = f'/content/drive/MyDrive/{folder_name}'

# Find all Excel files in the designated directory
excel_files = glob.glob(os.path.join(folder_path, '*.xlsx'))
print(f"Found Excel files: {excel_files}")

# Read and concatenate all 4 Excel files
if len(excel_files) > 0:
    df_list = [pd.read_excel(file) for file in excel_files]
    df = pd.concat(df_list, ignore_index=True)
    print(f"Successfully combined files. New dataset shape: {df.shape}")
else:
    print(f"No Excel files found at '{folder_path}'. Please ensure you have created the shortcut in your Drive with the correct name.")

Found Excel files: []
No Excel files found at '/content/drive/MyDrive/LendingRiskFolder'. Please ensure you have created the shortcut in your Drive with the correct name.


In [3]:
import pandas as pd
import glob
import os

print("Searching your mounted Google Drive for the lending risk Excel files...")
# Search recursively throughout the entire mounted MyDrive directory
all_drive_files = glob.glob('/content/drive/MyDrive/**/*.xlsx', recursive=True) + \
                  glob.glob('/content/drive/MyDrive/**/*.xls', recursive=True)

valid_files = []
for file in all_drive_files:
    try:
        # Read only the header of each excel file to see if it is the correct dataset
        header_df = pd.read_excel(file, nrows=2)
        if 'good_bad' in header_df.columns:
            valid_files.append(file)
            print(f"Found matching dataset: {os.path.basename(file)}")
    except Exception:
        continue

# Read and merge the found valid files
if len(valid_files) > 0:
    print(f"\nLoading and combining {len(valid_files)} matching files...")
    df_list = [pd.read_excel(file) for file in valid_files]
    df = pd.concat(df_list, ignore_index=True)
    print(f"\nSuccessfully combined files! Combined dataset shape: {df.shape}")
else:
    print("\nCould not find any Excel files with the column 'good_bad' in your Google Drive.")
    print("Please ensure you have added the shared folder to your Google Drive (e.g., by clicking 'Add shortcut to Drive' on the shared folder link).")

Searching your mounted Google Drive for the lending risk Excel files...
Found matching dataset: loan_data_preprocessed.xlsx
Found matching dataset: lending risk 1A.xlsx
Found matching dataset: lending risk 1B.xlsx
Found matching dataset: lending risk 2A.xlsx
Found matching dataset: lending risk 2B.xlsx

Loading and combining 5 matching files...

Successfully combined files! Combined dataset shape: (932570, 210)


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA INGESTION
# ==========================================
print("Using already loaded dataset...")
# We use the combined 'df' variable from the previous step
print(f"Dataset loaded. Shape: {df.shape}")

# ==========================================
# 2. TARGET DEFINITION & LEAKAGE PREVENTION
# ==========================================
# In your dataset, good_bad = 1 (Good), 0 (Bad).
# We flip it so 1 = Default, allowing models to predict Default Risk.
y = 1 - df['good_bad']

# Drop post-origination columns (Data Leakage) and ID/Target columns
leakage_and_id_cols = [
    's/n', 'id', 'member_id', 'good_bad', 'loan_status', 'url', 'desc', 'title',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'out_prncp', 'out_prncp_inv',
    'issue_d', 'earliest_cr_line', 'last_credit_pull_d' # Dates dropped for baseline simplicity
]

# Keep only columns that actually exist in the dataframe
cols_to_drop = [c for c in leakage_and_id_cols if c in df.columns]
df_clean = df.drop(columns=cols_to_drop)

# ==========================================
# 3. FEATURE SELECTION & PREPROCESSING
# ==========================================
# For this initial benchmark, we filter for numeric features
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
X = df_clean[numeric_cols]

# Drop columns that are completely empty (100% NaN) to prevent empty median imputation issues
X = X.dropna(axis=1, how='all')

# Impute missing values with the median, and fill any remaining edge-case NaNs with 0
medians = X.median()
X = X.fillna(medians).fillna(0)

# ==========================================
# 4. TRAIN / TEST SPLIT
# ==========================================
# Stratify ensures the Default rate remains exactly the same in Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining on {X_train.shape[0]} rows, Testing on {X_test.shape[0]} rows.")
print(f"Baseline Default Rate: {y.mean() * 100:.2f}%\n")

# ==========================================
# 5. SIMPLE ALGORITHM: LOGISTIC REGRESSION
# ==========================================
print("--- Training Logistic Regression (Baseline) ---")
lr = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)

# Predict probabilities
lr_preds = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_preds)
print(f"Logistic Regression ROC-AUC Score: {lr_auc:.4f}")

# ==========================================
# 6. DIFFICULT ALGORITHM: XGBOOST
# ==========================================
print("\n--- Training XGBoost (Challenger) ---")
# scale_pos_weight helps XGBoost handle the class imbalance naturally
scale_weight = (len(y_train) - y_train.sum()) / y_train.sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Predict probabilities
xgb_preds = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_preds)
print(f"XGBoost ROC-AUC Score: {xgb_auc:.4f}")

# ==========================================
# 7. FEATURE IMPORTANCE (XGBOOST)
# ==========================================
print("\nTop 5 Drivers of Default Risk (XGBoost):")
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance.head(5).to_string(index=False))

Using already loaded dataset...
Dataset loaded. Shape: (932570, 210)

Training on 746056 rows, Testing on 186514 rows.
Baseline Default Rate: 10.93%

--- Training Logistic Regression (Baseline) ---
Logistic Regression ROC-AUC Score: 0.6210

--- Training XGBoost (Challenger) ---
XGBoost ROC-AUC Score: 1.0000

Top 5 Drivers of Default Risk (XGBoost):
                                                        Feature  Importance
                                        loan_status:Charged Off    0.677209
                                 loan_status:Late (31-120 days)    0.245703
loan_status:Does not meet the credit policy. Status:Charged Off    0.039779
                                            loan_status:Default    0.037263
                                            loan_status:Current    0.000021
